# Land cover classification at the Mississppi Delta

In this notebook, you will use a k-means **unsupervised** clustering
algorithm to group pixels by similar spectral signatures. **k-means** is
an **exploratory** method for finding patterns in data. Because it is
unsupervised, you don’t need any training data for the model. You also
can’t measure how well it “performs” because the clusters will not
correspond to any particular land cover class. However, we expect at
least some of the clusters to be identifiable as different types of land
cover.

You will use the [harmonized Sentinal/Landsat multispectral
dataset](https://lpdaac.usgs.gov/documents/1698/HLS_User_Guide_V2.pdf).
You can access the data with an [Earthdata
account](https://www.earthdata.nasa.gov/learn/get-started) and the
[`earthaccess` library from
NSIDC](https://github.com/nsidc/earthaccess):

## STEP 1: SET UP

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Import all libraries you will need for this analysis</li>
<li>Configure GDAL parameters to help avoid connection errors:
<code>python      os.environ["GDAL_HTTP_MAX_RETRY"] = "5"      os.environ["GDAL_HTTP_RETRY_DELAY"] = "1"</code></li>
</ol></div></div>

In [2]:
## Cell 1: Import libraries / tools

import os # build file directories
import pickle # cache data objects, functions, or files
import re # Work with regular expressions
import warnings # prevent irrelevant warnings

import cartopy.crs as ccrs # cartography package, change coordinate reference systems
import earthaccess # access online data
import earthpy as et # python package for working with earth data
import geopandas as gpd # plots and data tables for geospatial data
import geoviews as gv # data visualization for geospatial data
import hvplot.pandas # interactive data plots 
import hvplot.xarray # interactive plots for raster data
import numpy as np # mathematical functions and analysis in python
import pandas as pd # work with data plots and tables
import rioxarray as rxr # work with rasters
import rioxarray.merge as rxrmerge # merge & mosaic rasters
from tqdm.notebook import tqdm # Use progress bars when downloading data or completing time-intensive tasks
import xarray as xr # work with rasters
from shapely.geometry import Polygon # work with geospatial polygons
from sklearn.cluster import KMeans # build K means cluster models

import pathlib # build paths and directories
import requests

os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "1"

warnings.simplefilter('ignore')

c:\Users\moenc\miniconda3\envs\earth-analytics-python\Lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
# # Data directory
# data_dir = os.path.join(
    
#     # Home directory
#     pathlib.Path.home(),
#     'EDA-Spring-2025',
#     'Clustering'
# )

Below you can find code for a caching **decorator** which you can use in
your code. To use the decorator:

``` python
@cached(key, override)
def do_something(*args, **kwargs):
    ...
    return item_to_cache
```

This decorator will **pickle** the results of running the
`do_something()` function, and only run the code if the results don’t
already exist. To override the caching, for example temporarily after
making changes to your code, set `override=True`. Note that to use the
caching decorator, you must write your own function to perform each
task!

In [4]:
## Cell 2: Define a function to "pickle" the data download (cache it on the computer)

def cached(func_key, override=False): # Defines a decorator called "cached", which is used to cache the results of function calls by saving them as pickle files.
    """
    A decorator to cache function results
    
    Parameters
    ==========
    key: str
      File basename used to save pickled results
    override: bool
      When True, re-compute even if the results are already stored
    """
    def compute_and_cache_decorator(compute_function): # 'wraps' the primary function in another function
        """
        Wrap the caching function
        
        Parameters
        ==========
        compute_function: function
          The function to run and cache results
        """
        def compute_and_cache(*args, **kwargs): # defines the primary function
            """
            Perform a computation and cache, or load cached result.
            
            Parameters
            ==========
            args
              Positional arguments for the compute function
            kwargs
              Keyword arguments for the compute function
            """
            # Add an identifier from the particular function call
            if 'cache_key' in kwargs:
                key = '_'.join((func_key, kwargs['cache_key']))
            else:
                key = func_key

            path = os.path.join(
                et.io.HOME, et.io.DATA_NAME, 'jars', f'{key}.pickle')
            
            # Check if the cache exists already or override caching
            if not os.path.exists(path) or override:
                # Make jars directory if needed
                os.makedirs(os.path.dirname(path), exist_ok=True)
                
                # Run the compute function as the user did
                result = compute_function(*args, **kwargs)
                
                # Pickle the object
                with open(path, 'wb') as file:
                    pickle.dump(result, file)
            else:
                # Unpickle the object
                with open(path, 'rb') as file:
                    result = pickle.load(file)
                    
            return result
        
        return compute_and_cache
    
    return compute_and_cache_decorator

## STEP 2: STUDY SITE

For this analysis, you will use a watershed from the [Water Boundary
Dataset](https://www.usgs.gov/national-hydrography/access-national-hydrography-products),
HU12 watersheds (WBDHU12.shp).

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Download the Water Boundary Dataset for region 8 (Mississippi)</li>
<li>Select watershed 080902030506</li>
<li>Generate a site map of the watershed</li>
</ol>
<p>Try to use the <strong>caching decorator</strong></p></div></div>

We chose this watershed because it covers parts of New Orleans an is
near the Mississippi Delta. Deltas are boundary areas between the land
and the ocean, and as a result tend to contain a rich variety of
different land cover and land use types.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-response"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div></div><div class="callout-body-container callout-body"><p>Write a 2-3 sentence <strong>site description</strong> (with
citations) of this area that helps to put your analysis in context.</p></div></div>

In [5]:
## Cell 3: Download the watershed boundary shapefile, and plot it with ESRI tiles in the background.

# Call cacheing function
@cached('wbd_08', override=False)
def read_wbd_file(wbd_filename, huc_level, cache_key): # Define a function to read the downloaded wbd shapefile
    
    # Define a template URL for downloading the zipped WBD dataset
    wbd_url = (
        "https://prd-tnm.s3.amazonaws.com"
        "/StagedProducts/Hydrography/WBD/HU2/Shape/"
        f"{wbd_filename}.zip")
    
    # Define wbd directory (use earthpy to get data)
    wbd_dir = et.data.get_data(url=wbd_url)

  # Read desired data
    wbd_path = os.path.join(wbd_dir, 'Shape', f'WBDHU{huc_level}.shp') # create a template to read the downloaded the wbd data as a shapefile, and a path to store the shapefile.
    wbd_gdf = gpd.read_file(wbd_path, engine='pyogrio') # read the wbd data as a gdf
    return wbd_gdf # print the gdf

# Define huc level
huc_level = 12

# Define wbd_gdf in terms of function defined previously (read_wbd_file), with args and kwargs as the WBD shapefile, huc_level, and cache_key.
wbd_gdf = read_wbd_file(
    "WBD_08_HU2_Shape", huc_level, cache_key=f'hu{huc_level}')

# Define delta_gdf in terms of wbd_gdf, huc level, and watershed identifier. dissolve borders of the shapefile.
delta_gdf = (
    wbd_gdf[wbd_gdf[f'huc{huc_level}']
    .isin(['080902030506'])]
    .dissolve()
)

(
    # Define coordinate reference system for delta_gdf as Mercator
    delta_gdf.to_crs(ccrs.Mercator())
    # Make an interactive hv plot, with ESRI tiles in the background.
    .hvplot(
        alpha=.2, fill_color='white', 
        tiles='EsriImagery', crs=ccrs.Mercator(), title="Mississippi River Delta Subwatershed 08")
    .opts(width=600, height=300)
)


:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

Site Description:

The Mississippi River Delta is an area of exceptional biodiversity, with diverse ecosystems and land use characteristics. According to the National Wildlife Foundation, the delta (Louisiana, in particular) contains 40% of the coastal wetlands for the entire lower 48 states. (Source 1)  These wetlands, and the diverse environments of the delta, are critically important as habitat for birds, productive agricultural land (due to the silt from the Mississippi River), as well as supporting productive fisheries. However, these diverse habitats are under threat both from natural erosion and from human activities. The particular subwatershed in question represents an area South of New Orleans, which stretches from about 29.82000N-29.68000N (Latitude) to -89.97000W--89.78000W (Longitude). 

Sources:

1) https://www.nwf.org/Educational-Resources/Wildlife-Guide/Wild-Places/Mississippi-River-Delta
2) https://www.nps.gov/miss/riverfacts.htm#:~:text=Some%20-like%20to%20measure%20the,(27%20million%20square%20miles).
3) https://nps.gov/locations/lowermsdeltaregion/the-natural-environment-the-delta-and-its-resources.htm
4) https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/6
5) https://www.ars.usda.gov/ARSUserFiles/np211/LMRBProposal.pdf
6) https://www.lsu.edu/lgs/publications/products/landforms_book.pdf
7) https://pubs.usgs.gov/of/2009/1280/pdf/of2009-1280.pdf 

Data citation:
https://www.usgs.gov/national-hydrography/access-national-hydrography-products

## STEP 3: MULTISPECTRAL DATA

### Search for data

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Log in to the <code>earthaccess</code> service using your Earthdata
credentials:
<code>python      earthaccess.login(persist=True)</code></li>
<li>Modify the following sample code to search for granules of the
HLSL30 product overlapping the watershed boundary from May to October
2023 (there should be 76 granules):
<code>python      results = earthaccess.search_data(          short_name="...",          cloud_hosted=True,          bounding_box=tuple(gdf.total_bounds),          temporal=("...", "..."),      )</code></li>
</ol></div></div>

In [6]:
## Cell 4: login to earthaccess, and search for HLS tiles covering the WBD site.

# Log in to earthaccess
earthaccess.login(persist=True)

# Search for HLS tiles
results = earthaccess.search_data(
    short_name="HLSL30",
    cloud_hosted=True, 
    bounding_box=tuple(delta_gdf.total_bounds), # tiles bounded by the delta_gdf (wbd file)
    temporal=("2024-06", "2024-08"), # date range
)
results

[Collection: {'EntryTitle': 'HLS Landsat Operational Land Imager Surface Reflectance and TOA Brightness Daily Global 30m v2.0'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Longitude': -89.79864173, 'Latitude': 29.70347853}, {'Longitude': -89.76643746, 'Latitude': 30.69278312}, {'Longitude': -90.91181412, 'Latitude': 30.71627038}, {'Longitude': -90.93262544, 'Latitude': 29.72659663}, {'Longitude': -89.79864173, 'Latitude': 29.70347853}]}}]}}}
 Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2024-06-07T16:31:11.509Z', 'EndingDateTime': '2024-06-07T16:31:11.509Z'}}
 Size(MB): 169.50417041778564
 Data: ['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B10.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.SAA.tif', 'https://data.l

### Compile information about each granule

I recommend building a GeoDataFrame, as this will allow you to plot the
granules you are downloading and make sure they line up with your
shapefile. You could also use a DataFrame, dictionary, or a custom
object to store this information.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>For each search result:
<ol type="1">
<li>Get the following information (HINT: look at the [‘umm’] values for
each search result):
<ul>
<li>granule id (UR)</li>
<li>datetime</li>
<li>geometry (HINT: check out the shapely.geometry.Polygon class to
convert points to a Polygon)</li>
</ul></li>
<li>Open the granule files. I recomment opening one granule at a time,
e.g. with (<code>earthaccess.open([result]</code>).</li>
<li>For each file (band), get the following information:
<ul>
<li>file handler returned from <code>earthaccess.open()</code></li>
<li>tile id</li>
<li>band number</li>
</ul></li>
</ol></li>
<li>Compile all the information you collected into a GeoDataFrame</li>
</ol></div></div>

In [7]:
## Cell 5: Define a function to compile data access links from earthaccess (HLS tiles covering the WBD region)

def get_earthaccess_links(results): # Name the function "get_earthaccess_links"
    url_re = re.compile( # compile regular expressions
        r'\.(?P<tile_id>\w+)\.\d+T\d+\.v\d\.\d\.(?P<band>[A-Za-z0-9]+)\.tif' # regular expression pattern for .tiff data downloads
    )

    # Loop through each granule
    link_rows = [] # Create an empty list to put the links in
    for granule in tqdm(results):
        # Get granule information
        info_dict = granule['umm'] # Creates a list of granule metadata values
        granule_id = info_dict['GranuleUR'] # List of granule identifiers
        datetime = pd.to_datetime(
            info_dict['TemporalExtent']['RangeDateTime']['BeginningDateTime'] # creates sublists within the info_dict which identify the date and time information for the granules
        )
        points = (
            info_dict['SpatialExtent']['HorizontalSpatialDomain']['Geometry']
            ['GPolygons'][0]['Boundary']['Points'] # sublists within info_dict define spatial and geometric characteristics of each granule
        )
        geometry = Polygon([(point['Longitude'], point['Latitude']) for point in points]) # This line is running some code from the "Polygon" library. Looks like it's using the granule spatial info gathered in "points" to create polygons.

        # Get URLs for each granule
        files = earthaccess.open([granule])

        # Build metadata DataFrame rows
        for file in files: # set up loop
            match = url_re.search(file.full_name) # define 'match' as the full_name of each granule file
            if match is not None: # if match exists
                link_rows.append( # append to link_rows
                    gpd.GeoDataFrame( # (what to append to link_rows): a gdf for each granule, complete with datetime, tile_id, band, url, and geometry, with CRS = EPSG:4326.
                        dict(
                            datetime=[datetime],
                            tile_id=[match.group('tile_id')],
                            band=[match.group('band')],
                            url=[file],
                            geometry=[geometry]
                        ),
                        crs="EPSG:4326" # Set coordinate reference system
                    )
                )

    # Concatenate metadata DataFrame (compile everything into a gdf)
    if link_rows:
        file_df = pd.concat(link_rows).reset_index(drop=True) # define "file_df" by concatenating and resetting the index (this is merging the granules / HLS tiles into one tile and resetting the index).
    else:
        file_df = gpd.GeoDataFrame(columns=["datetime", "tile_id", "band", "url", "geometry"], crs="EPSG:4326")

    return file_df


### Open, crop, and mask data

This will be the most resource-intensive step. I recommend caching your
results using the `cached` decorator or by writing your own caching
code. I also recommend testing this step with one or two dates before
running the full computation.

This code should include at least one **function** including a
numpy-style docstring. A good place to start would be a function for
opening a single masked raster, applying the appropriate scale
parameter, and cropping.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>For each granule:
<ol type="1">
<li><p>Open the Fmask band, crop, and compute a quality mask for the
granule. You can use the following code as a starting point, making sure
that <code>mask_bits</code> contains the quality bits you want to
consider: ```python # Expand into a new dimension of binary bits bits =
( np.unpackbits(da.astype(np.uint8), bitorder=‘little’)
.reshape(da.shape + (-1,)) )</p>
<p># Select the required bits and check if any are flagged mask =
np.prod(bits[…, mask_bits]==0, axis=-1) ```</p></li>
<li><p>For each band that starts with ‘B’:</p>
<ol type="1">
<li>Open the band, crop, and apply the scale factor</li>
<li>Name the DataArray after the band using the <code>.name</code>
attribute</li>
<li>Apply the cloud mask using the <code>.where()</code> method</li>
<li>Store the DataArray in your data structure (e.g. adding a
GeoDataFrame column with the DataArray in it. Note that you will need to
remove the rows for unused bands)</li>
</ol></li>
</ol></li>
</ol></div></div>

In [8]:
## Cell 6: Process the HLS data, looping through each granule to add a cloud mask, crop, and define scale.

@cached('delta_reflectance_da_df')
def compute_reflectance_da(search_results, boundary_gdf):
    """
    Connect to files over VSI, crop, cloud mask, and wrangle
    
    Returns a single reflectance DataFrame 
    with all bands as columns and
    centroid coordinates and datetime as the index.
    
    Parameters
    ==========
    file_df : pd.DataFrame
        File connection and metadata (datetime, tile_id, band, and url)
    boundary_gdf : gpd.GeoDataFrame
        Boundary use to crop the data
    """
    def open_dataarray(url, boundary_proj_gdf, scale=1, masked=True):
        # Open masked DataArray
        da = rxr.open_rasterio(url, masked=masked).squeeze() * scale
        
        # Reproject boundary if needed
        if boundary_proj_gdf is None:
            boundary_proj_gdf = boundary_gdf.to_crs(da.rio.crs)
            
        # Crop
        cropped = da.rio.clip_box(*boundary_proj_gdf.total_bounds)
        return cropped
    
    def compute_quality_mask(da, mask_bits=[1, 2, 3]):
        """Mask out low quality data by bit"""
        # Unpack bits into a new axis
        bits = (
            np.unpackbits(
                da.astype(np.uint8), bitorder='little'
            ).reshape(da.shape + (-1,))
        )

        # Select the required bits and check if any are flagged
        mask = np.prod(bits[..., mask_bits]==0, axis=-1)
        return mask

    file_df = get_earthaccess_links(search_results)
    
    granule_da_rows= []
    boundary_proj_gdf = None

    # Loop through each image
    group_iter = file_df.groupby(['datetime', 'tile_id'])
    for (datetime, tile_id), granule_df in tqdm(group_iter):
        print(f'Processing granule {tile_id} {datetime}')
              
        # Open granule cloud cover
        cloud_mask_url = (
            granule_df.loc[granule_df.band=='Fmask', 'url']
            .values[0])
        cloud_mask_cropped_da = open_dataarray(cloud_mask_url, boundary_proj_gdf, masked=False)

        # Compute cloud mask
        cloud_mask = compute_quality_mask(cloud_mask_cropped_da)

        # Loop through each spectral band
        da_list = []
        df_list = []
        for i, row in granule_df.iterrows():
            if row.band.startswith('B'):
                # Open, crop, and mask the band
                band_cropped = open_dataarray(
                    row.url, boundary_proj_gdf, scale=0.0001)
                band_cropped.name = row.band
                # Add the DataArray to the metadata DataFrame row
                row['da'] = band_cropped.where(cloud_mask)
                granule_da_rows.append(row.to_frame().T)
    
    # Reassemble the metadata DataFrame
    return pd.concat(granule_da_rows)

reflectance_da_df = compute_reflectance_da(results, delta_gdf)

### Merge and Composite Data

You will notice for this watershed that: 1. The raster data for each
date are spread across 4 granules 2. Any given image is incomplete
because of clouds

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li><p>For each band:</p>
<ol type="1">
<li><p>For each date:</p>
<ol type="1">
<li>Merge all 4 granules</li>
<li>Mask any negative values created by interpolating from the nodata
value of -9999 (<code>rioxarray</code> should account for this, but
doesn’t appear to when merging. If you leave these values in they will
create problems down the line)</li>
</ol></li>
<li><p>Concatenate the merged DataArrays along a new date
dimension</p></li>
<li><p>Take the mean in the date dimension to create a composite image
that fills cloud gaps</p></li>
<li><p>Add the band as a dimension, and give the DataArray a
name</p></li>
</ol></li>
<li><p>Concatenate along the band dimension</p></li>
</ol></div></div>

In [9]:
## Cell 7: Create a function that merges granules across dates, filling in cloud gaps and masking negative values.

@cached('delta_reflectance_da')
def merge_and_composite_arrays(granule_da_df):
    # Merge and composite and image for each band
    df_list = []
    da_list = []
    for band, band_df in tqdm(granule_da_df.groupby('band')):
        merged_das = []
        for datetime, date_df in tqdm(band_df.groupby('datetime')):
            # Merge granules for each date
            merged_da = rxrmerge.merge_arrays(list(date_df.da))
            # Mask negative values
            merged_da = merged_da.where(merged_da>0)
            merged_das.append(merged_da)
            
        # Composite images across dates
        composite_da = xr.concat(merged_das, dim='datetime').median('datetime')
        composite_da['band'] = int(band[1:])
        composite_da.name = 'reflectance'
        da_list.append(composite_da)
        
    return xr.concat(da_list, dim='band')

reflectance_da = merge_and_composite_arrays(reflectance_da_df)
reflectance_da

<xarray.DataArray 'reflectance' (band: 10, y: 556, x: 624)> Size: 14MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], dtype=float32)
Coordinates:
  * x            (x) float64 5kB 7.926e+05 7.926e+05 ... 8.112e+05 8.113e+05
  * y            (y) float64 4kB 3.304e+06 3.304e+06 ... 3.287e+06 3.287e+06
  * band         (band) int64 80B 1 2 3 4 5 6 7 9 10 11
    spatial_ref  int64 8B 0

## STEP 4: K-MEANS

Cluster your data by spectral signature using the k-means algorithm.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Convert your DataArray into a <strong>tidy</strong> DataFrame of
reflectance values (hint: check out the <code>.to_dataframe()</code> and
<code>.unstack()</code> methods)</li>
<li>Filter out all rows with no data (all 0s or any N/A values)</li>
<li>Fit a k-means model. You can experiment with the number of groups to
find what works best.</li>
</ol></div></div>

In [10]:
## Cell 8: Clean up the dataframe (.dropna) and fit KMeans model.

# Convert spectral DataArray to a tidy DataFrame
model_df = reflectance_da.to_dataframe().reflectance.unstack('band')
model_df = model_df.drop(columns=[10, 11]).dropna()

# Running the fit and predict functions at the same time.
# We can do this since we don't have target data.
prediction = KMeans(n_clusters=6).fit_predict(model_df.values)

# Add the predicted values back to the model DataFrame
model_df['clusters'] = prediction
model_df

band                              1       2       3       4       5       6  \
y            x                                                                
3.303783e+06 810148.062907  0.01560  0.0225  0.0409  0.0366  0.0478  0.0281   
             810178.062907  0.01895  0.0256  0.0396  0.0413  0.0426  0.0284   
             810208.062907  0.01915  0.0246  0.0387  0.0377  0.0384  0.0273   
             810238.062907  0.02040  0.0247  0.0440  0.0445  0.0629  0.0418   
             810268.062907  0.01815  0.0245  0.0437  0.0444  0.0618  0.0397   
...                             ...     ...     ...     ...     ...     ...   
3.287163e+06 793798.062907  0.02650  0.0345  0.0548  0.0427  0.0218  0.0098   
             793828.062907  0.02790  0.0351  0.0549  0.0439  0.0221  0.0104   
             793858.062907  0.02580  0.0331  0.0534  0.0419  0.0194  0.0080   
             793888.062907  0.02570  0.0326  0.0521  0.0402  0.0182  0.0064   
             793918.062907  0.02550  0.0340  0.0541  0.0423  0.0199  0.0083   

band                             7       9  clusters  
y            x                                        
3.303783e+06 810148.062907  0.0181  0.0006         0  
             810178.062907  0.0220  0.0006         0  
             810208.062907  0.0236  0.0007         0  
             810238.062907  0.0266  0.0007         0  
             810268.062907  0.0259  0.0008         0  
...                            ...     ...       ...  
3.287163e+06 793798.062907  0.0074  0.0007         0  
             793828.062907  0.0076  0.0008         0  
             793858.062907  0.0059  0.0009         0  
             793888.062907  0.0046  0.0007         0  
             793918.062907  0.0060  0.0007         0  

[318248 rows x 9 columns]

## STEP 5: PLOT

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Create a plot that shows the k-means clusters next to an RGB image of
the area. You may need to brighten your RGB image by multiplying it by
10. The code for reshaping and plotting the clusters is provided for you
below, but you will have to create the RGB plot yourself!</p>
<p>So, what is <code>.sortby(['x', 'y'])</code> doing for us? Try the
code without it and find out.</p></div></div>

In [17]:
## Cell 9: Plot K-means and RGB images of the watershed.

# Plot the k-means clusters
rgb = reflectance_da.sel(band=[4, 3, 2])
rgb_uint8 = (rgb * 255).astype(np.uint8).where(rgb!=np.nan)
rgb_bright = rgb_uint8 * 10
rgb_sat = rgb_bright.where(rgb_bright < 255, 255)

(
    rgb_sat.hvplot.rgb( 
        x='x', y='y', bands='band',
        data_aspect=1,
        xaxis=None, yaxis=None, title="K-means and RGB comparison")
    + 
    model_df.clusters.to_xarray().sortby(['x', 'y']).hvplot(
        cmap="accent", aspect='equal') 
)

:Layout
   .RGB.I   :RGB   [x,y]   (R,G,B)
   .Image.I :Image   [x,y]   (clusters)

In [12]:
# initialize
k_means = KMeans(n_clusters = 5)

# fit model and predict
prediction = k_means.fit_predict(model_df.values)

model_df['clusters'] = prediction
model_df

band                              1       2       3       4       5       6  \
y            x                                                                
3.303783e+06 810148.062907  0.01560  0.0225  0.0409  0.0366  0.0478  0.0281   
             810178.062907  0.01895  0.0256  0.0396  0.0413  0.0426  0.0284   
             810208.062907  0.01915  0.0246  0.0387  0.0377  0.0384  0.0273   
             810238.062907  0.02040  0.0247  0.0440  0.0445  0.0629  0.0418   
             810268.062907  0.01815  0.0245  0.0437  0.0444  0.0618  0.0397   
...                             ...     ...     ...     ...     ...     ...   
3.287163e+06 793798.062907  0.02650  0.0345  0.0548  0.0427  0.0218  0.0098   
             793828.062907  0.02790  0.0351  0.0549  0.0439  0.0221  0.0104   
             793858.062907  0.02580  0.0331  0.0534  0.0419  0.0194  0.0080   
             793888.062907  0.02570  0.0326  0.0521  0.0402  0.0182  0.0064   
             793918.062907  0.02550  0.0340  0.0541  0.0423  0.0199  0.0083   

band                             7       9  clusters  
y            x                                        
3.303783e+06 810148.062907  0.0181  0.0006         1  
             810178.062907  0.0220  0.0006         1  
             810208.062907  0.0236  0.0007         1  
             810238.062907  0.0266  0.0007         1  
             810268.062907  0.0259  0.0008         1  
...                            ...     ...       ...  
3.287163e+06 793798.062907  0.0074  0.0007         1  
             793828.062907  0.0076  0.0008         1  
             793858.062907  0.0059  0.0009         1  
             793888.062907  0.0046  0.0007         1  
             793918.062907  0.0060  0.0007         1  

[318248 rows x 9 columns]

In [13]:
# # make a data array with bands to use for rgb
# rgb = reflectance_da.sel(band=[4, 3, 2])

In [14]:
# # plot it
# (
#     rgb.hvplot.rgb(y = 'y',
#                    x = 'x',
#                    data_aspect = 1,
#                    xaxis = None, yaxis = None)
# )

In [15]:
# want image pixel values to range from 0-255 (instead of 0-1)
# then convert to unsigned 88-bit integers
# don't include NAs in calculation
# rgb_unit8_bright=

In [16]:
# # Plot the k-means clusters
# (
#     rgb_plot
#     + 
#     model_df.clusters.to_xarray().sortby(['x', 'y']).hvplot(
#         cmap="Colorblind", aspect='equal') 
# )

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-respond"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Reflect and Respond</div></div><div class="callout-body-container callout-body"><p>Don’t forget to interpret your plot!</p></div></div>

**YOUR PLOT HEADLINE AND DESCRIPTION HERE**

The k-means plot groups the data based on similar reflectance characteristics. The assumption is that areas with similar reflectance characteristics have similar vegetation and land-use patterns. 

The K-means plot creates groups based on similar reflectance characteristics. 